# 📗 Notebook 2: Advanced Topic Modeling
## Online LDA · Guided LDA · GSDMM · STM · DTM · Author-Topic — BSAN 6200

**What you'll learn:**
- Measure topic quality beyond coherence: **diversity** and **stability**
- Handle streaming/large data with **Online LDA**
- Inject domain knowledge with **Guided/Seeded LDA**
- Model short texts (tweets, titles) with **GSDMM**
- Connect topics to metadata with **Structural Topic Model (STM)**
- Track topics over time with **Dynamic Topic Model (DTM)**
- Discover author-level topic profiles with **Author-Topic Model**

**Dataset:** 20 Newsgroups (4 categories)  
**Prerequisite:** Complete Notebook 1 first (preprocessing concepts carry over)

## 0. Install & Imports

In [ ]:
!pip install gensim gsdmm tomotopy guidedlda -q

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import spacy
from collections import Counter, defaultdict

# Gensim
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel, AuthorTopicModel
from gensim.models.ldamulticore import LdaMulticore

# Sklearn
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Viz
from wordcloud import WordCloud

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    import subprocess
    subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'])
    nlp = spacy.load('en_core_web_sm')

print('✅ Core imports ready')

## 1. Load & Preprocess Dataset

In [ ]:
CATEGORIES = ['sci.space', 'rec.sport.baseball',
              'talk.politics.guns', 'comp.graphics']

newsgroups = fetch_20newsgroups(
    subset='all',
    categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

STOP_WORDS = set(stopwords.words('english'))
EXTRA_STOPS = {
    'would', 'could', 'like', 'think', 'know', 'get', 'one',
    'say', 'use', 'also', 'even', 'may', 'see', 'well', 'way',
    'make', 'take', 'come', 'good', 'new', 'time', 'year',
    'people', 'write', 'go', 'right', 'thing', 'want', 'need',
}
STOP_WORDS |= EXTRA_STOPS

def preprocess(text):
    text = re.sub(r'\S+@\S+|http\S+|\d+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    doc = nlp(text.lower(), disable=['parser', 'ner'])
    return [
        token.lemma_ for token in doc
        if token.lemma_ not in STOP_WORDS
        and token.is_alpha and len(token.lemma_) > 2
    ]

print('Preprocessing... (~60s)')
documents    = newsgroups.data
labels       = newsgroups.target
label_names  = newsgroups.target_names

tokenized    = [preprocess(d) for d in documents]
mask         = [i for i, t in enumerate(tokenized) if len(t) >= 5]
tokenized    = [tokenized[i] for i in mask]
documents    = [documents[i] for i in mask]
labels       = [labels[i]    for i in mask]

dictionary = Dictionary(tokenized)
dictionary.filter_extremes(no_below=5, no_above=0.80)
bow_corpus = [dictionary.doc2bow(t) for t in tokenized]

print(f'✅ {len(documents)} docs, vocab size {len(dictionary):,}')

---
## 2. Topic Quality Metrics

Beyond coherence, we need two more checks:
1. **Topic Diversity** — are topics genuinely different from each other?
2. **Topic Stability** — does the model produce the same topics across different random seeds?

### 2.1 Topic Diversity

In [ ]:
def topic_diversity(model, topn=10):
    """
    Diversity = fraction of unique words across all topic top-N word lists.
    Range: 0 to 1. Higher = better (less redundancy between topics).
    
    Rule of thumb: aim for diversity > 0.7
    """
    all_words = []
    for i in range(model.num_topics):
        top_words = [w for w, _ in model.show_topic(i, topn=topn)]
        all_words.extend(top_words)
    return len(set(all_words)) / len(all_words)

# Train a baseline model for demonstration
base_model = LdaModel(
    corpus=bow_corpus, id2word=dictionary,
    num_topics=4, random_state=42, passes=10
)
div = topic_diversity(base_model)
print(f'Topic Diversity: {div:.3f}')
if div < 0.7:
    print('⚠️  Low diversity — topics may be redundant. Consider reducing k.')
else:
    print('✅ Good diversity — topics are meaningfully distinct.')

In [ ]:
# ── Visualize overlap with a heatmap ─────────────────────────────
def topic_word_overlap_matrix(model, topn=15):
    """Jaccard similarity matrix between all pairs of topics."""
    n = model.num_topics
    topics = [set(w for w, _ in model.show_topic(i, topn=topn)) for i in range(n)]
    mat = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            mat[i][j] = len(topics[i] & topics[j]) / len(topics[i] | topics[j])
    return mat

overlap = topic_word_overlap_matrix(base_model)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(overlap, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=[f'T{i}' for i in range(4)],
            yticklabels=[f'T{i}' for i in range(4)],
            ax=ax)
ax.set_title('Topic Word Overlap (Jaccard Similarity)\nDiagonal = 1.0 always', fontsize=12)
plt.tight_layout()
plt.show()
print('Off-diagonal values close to 0 = good topic separation.')

### 2.2 Topic Stability

In [ ]:
def jaccard(set1, set2):
    """Jaccard similarity between two sets."""
    return len(set1 & set2) / len(set1 | set2) if (set1 | set2) else 0.0

def topic_stability(corpus, dictionary, k=4, n_runs=5, topn=10):
    """
    Train the model n_runs times with different random seeds.
    For each pair of runs, find the best topic alignment and 
    compute average Jaccard similarity.
    Stability > 0.5 = reasonably stable.
    """
    run_topics = []
    for seed in range(n_runs):
        m = LdaModel(
            corpus=corpus, id2word=dictionary,
            num_topics=k, random_state=seed, passes=8
        )
        tops = [set(w for w, _ in m.show_topic(i, topn=topn)) for i in range(k)]
        run_topics.append(tops)

    # Compare run 0 vs all other runs — greedy best matching per topic
    base = run_topics[0]
    all_scores = []
    for run in run_topics[1:]:
        matched = set()
        scores = []
        for t_base in base:
            best = max([(jaccard(t_base, t_other), j)
                        for j, t_other in enumerate(run) if j not in matched])
            scores.append(best[0])
            matched.add(best[1])
        all_scores.append(np.mean(scores))

    return np.mean(all_scores), np.std(all_scores)

print('Running 5 seeds for k=4... (takes ~1min)')
mean_stab, std_stab = topic_stability(bow_corpus, dictionary, k=4, n_runs=5)
print(f'Topic Stability: {mean_stab:.3f} ± {std_stab:.3f}')
if mean_stab > 0.5:
    print('✅ Stable model — topics consistent across random seeds')
else:
    print('⚠️  Unstable — consider more passes or a different k')

---
## 3. Online / Incremental LDA

**Use when:** corpus is too large for memory, or new documents arrive continuously.  
Online LDA processes mini-batches and updates the model without full retraining.

Key parameters:
- `chunksize` — mini-batch size
- `passes=1` — only 1 pass over each batch (like stochastic gradient descent)
- `update_every=1` — update after each chunk

In [ ]:
# ── Split corpus into initial + streaming batches ─────────────────
initial_idx  = int(0.6 * len(bow_corpus))
stream_batch = 200  # simulate arriving in batches of 200

initial_corpus = bow_corpus[:initial_idx]
stream_corpus  = bow_corpus[initial_idx:]

print(f'Initial corpus: {len(initial_corpus)} docs')
print(f'Streaming:      {len(stream_corpus)} docs in batches of {stream_batch}')

In [ ]:
# ── Train initial Online LDA ─────────────────────────────────────
online_lda = LdaModel(
    corpus=initial_corpus,
    id2word=dictionary,
    num_topics=4,
    random_state=42,
    passes=1,         # 1 pass per batch = online learning
    update_every=1,
    chunksize=100,
    alpha='symmetric',
    eta='auto'
)
print('Initial Online LDA trained')

# Track coherence before update
cm_before = CoherenceModel(
    model=online_lda, texts=tokenized[:initial_idx],
    dictionary=dictionary, coherence='c_v'
)
coh_before = cm_before.get_coherence()
print(f'Coherence before streaming: {coh_before:.4f}')

In [ ]:
# ── Simulate streaming: update model with new batches ─────────────
n_batches = 0
for start in range(0, len(stream_corpus), stream_batch):
    batch = stream_corpus[start:start + stream_batch]
    online_lda.update(batch)
    n_batches += 1

print(f'Updated with {n_batches} streaming batches')

cm_after = CoherenceModel(
    model=online_lda, texts=tokenized,
    dictionary=dictionary, coherence='c_v'
)
coh_after = cm_after.get_coherence()
print(f'Coherence after streaming:  {coh_after:.4f}')
print(f'Change: {coh_after - coh_before:+.4f}')

In [ ]:
# ── Compare topics before/after streaming ─────────────────────────
print('Topics after streaming update:')
for i in range(4):
    words = ', '.join([w for w, _ in online_lda.show_topic(i, 8)])
    print(f'  Topic {i}: {words}')

---
## 4. Guided / Seeded LDA

**Use when:** you have domain knowledge about expected topics.  
You provide seed words to steer the model — the rest of the corpus still explores freely.

This is semi-supervised: human knowledge + data-driven learning.

In [ ]:
try:
    import guidedlda
    print('✅ guidedlda available')
    GUIDED_AVAILABLE = True
except ImportError:
    print('⚠️  guidedlda not installed — showing code structure only')
    print('     Install with: pip install guidedlda')
    GUIDED_AVAILABLE = False

In [ ]:
# ── Build vocabulary and DTM for guidedlda ────────────────────────
from sklearn.feature_extraction.text import CountVectorizer

cleaned_texts = [' '.join(t) for t in tokenized]

cv = CountVectorizer(
    max_features=5000, min_df=5, max_df=0.80
)
dtm = cv.fit_transform(cleaned_texts)
vocab = cv.get_feature_names_out()

print(f'DTM shape: {dtm.shape}')
print(f'Vocabulary: {len(vocab):,} words')

In [ ]:
# ── Define seed words (domain knowledge) ──────────────────────────
# Keys are your topic names, values are lists of strongly indicative words
seed_topics = {
    'Space & Astronomy':    ['space', 'nasa', 'orbit', 'launch', 'satellite', 'moon', 'planet'],
    'Baseball & Sports':    ['game', 'player', 'team', 'pitch', 'bat', 'run', 'inning', 'win'],
    'Gun Politics & Rights':['gun', 'weapon', 'fire', 'law', 'government', 'right', 'amendment'],
    'Computer Graphics':    ['image', 'pixel', 'software', 'program', 'graphics', 'render', 'display'],
}

print('Seed words by topic:')
for name, words in seed_topics.items():
    print(f'  {name}: {words}')

In [ ]:
# ── Train Guided LDA ──────────────────────────────────────────────
if GUIDED_AVAILABLE:
    # Map seed words to vocabulary indices
    word2id = {w: i for i, w in enumerate(vocab)}
    seed_confidence = 0.15  # how strongly to enforce seeds (0.0 = ignore, 1.0 = force)

    guided_model = guidedlda.GuidedLDA(
        n_topics=4,
        n_iter=200,
        random_state=42,
        refresh=20
    )

    # Build seed-word matrix: shape (vocab_size, n_topics)
    topic_names = list(seed_topics.keys())
    seed_matrix = np.zeros((len(vocab), len(topic_names)))
    for t_idx, (name, words) in enumerate(seed_topics.items()):
        for w in words:
            if w in word2id:
                seed_matrix[word2id[w], t_idx] = seed_confidence

    guided_model.fit(dtm, seed_topics=seed_matrix, seed_confidence=seed_confidence)
    print('✅ Guided LDA trained')

    print('\nTopics found:')
    for i, name in enumerate(topic_names):
        top_words = [vocab[j] for j in guided_model.components_[i].argsort()[::-1][:10]]
        print(f'  {name}: {", ".join(top_words)}')

else:
    # Show what the code structure looks like even without the package
    print("""
# With guidedlda installed, the workflow is:
# 1. Build CountVectorizer DTM (already done above)
# 2. Map seed words to vocab indices
# 3. Build seed_matrix of shape (vocab_size, n_topics)
# 4. guided_model.fit(dtm, seed_topics=seed_matrix, seed_confidence=0.15)
# 5. Inspect topics — they'll be anchored to your seed words

# Compare to unsupervised LDA output to see the effect:
# Seeded topics are more predictable and explainable to stakeholders
""")

---
## 5. GSDMM — Short Text Topic Modeling

**Problem:** LDA assumes each document contains many words — tweets and titles have ~10 words.  
That's too few for co-occurrence statistics to work reliably.

**GSDMM** (Gibbs Sampling Dirichlet Multinomial Mixture):
- Key assumption: each document belongs to **one topic** (not a mixture)
- Works much better for short texts
- Similar to LDA but with one-topic-per-doc

In [ ]:
# ── Create a short text corpus (simulate tweets/headlines) ────────
# Extract first sentence of each document as a "short text"
def extract_headline(text, max_words=15):
    """Take first ~15 meaningful tokens as a synthetic short text."""
    tokens = text.split()
    return ' '.join(tokens[:max_words])

short_texts = [extract_headline(d) for d in documents]

# Tokenize the short texts
tokenized_short = [preprocess(t) for t in short_texts]
tokenized_short = [t for t in tokenized_short if len(t) >= 2]  # keep if ≥ 2 tokens

print(f'Short text corpus: {len(tokenized_short)} documents')
print(f'Avg tokens: {np.mean([len(t) for t in tokenized_short]):.1f}')
print()
print('Examples:')
for i in range(3):
    print(f'  {i}: {tokenized_short[i]}')

In [ ]:
# ── Build vocabulary for GSDMM ────────────────────────────────────
from gensim.corpora import Dictionary as GensimDict

short_dict = GensimDict(tokenized_short)
short_dict.filter_extremes(no_below=2, no_above=0.9)

vocab_gsdmm = list(short_dict.token2id.keys())
vocab_size   = len(vocab_gsdmm)
word2id_gsdmm = {w: i for i, w in enumerate(vocab_gsdmm)}

print(f'Short text vocab: {vocab_size:,} unique tokens')

In [ ]:
# ── Import GSDMM ─────────────────────────────────────────────────
try:
    from gsdmm import MovieGroupProcess
    print('✅ gsdmm available')
    GSDMM_AVAILABLE = True
except ImportError:
    print('⚠️  gsdmm not installed — install with: pip install gsdmm')
    GSDMM_AVAILABLE = False

In [ ]:
if GSDMM_AVAILABLE:
    mgp = MovieGroupProcess(
        K=8,      # max number of clusters (GSDMM will use fewer)
        alpha=0.1,
        beta=0.1,
        n_iters=30
    )

    # Convert tokenized docs to word IDs
    docs_as_ids = [
        [word2id_gsdmm[w] for w in doc if w in word2id_gsdmm]
        for doc in tokenized_short
    ]
    # Filter empty docs
    docs_as_ids = [d for d in docs_as_ids if len(d) > 0]

    y = mgp.fit(docs_as_ids, vocab_size)

    # Check cluster population
    cluster_counts = Counter(y)
    active_clusters = {k: v for k, v in cluster_counts.items() if v > 5}
    print(f'Active clusters (>5 docs): {len(active_clusters)}')
    print('Cluster sizes:')
    for cid, cnt in sorted(active_clusters.items(), key=lambda x: -x[1]):
        print(f'  Cluster {cid}: {cnt} docs')

else:
    print("""
# GSDMM usage pattern:
# mgp = MovieGroupProcess(K=10, alpha=0.1, beta=0.1, n_iters=30)
# y = mgp.fit(docs_as_word_ids, vocab_size)
# cluster_doc_count = mgp.cluster_doc_count   # docs per cluster
# cluster_word_count = mgp.cluster_word_count  # word counts per cluster
""")

In [ ]:
if GSDMM_AVAILABLE and active_clusters:
    # ── Show top words per active cluster ────────────────────────────
    print('Top words per GSDMM cluster:')
    for cid in sorted(active_clusters.keys()):
        word_counts = mgp.cluster_word_count[cid]
        total = sum(word_counts.values())
        if total == 0:
            continue
        top_words = sorted(word_counts, key=word_counts.get, reverse=True)[:10]
        top_words = [vocab_gsdmm[j] for j in top_words if j < len(vocab_gsdmm)]
        print(f'  Cluster {cid}: {", ".join(top_words)}')

---
## 6. Structural Topic Model (STM) — Metadata-Aware Topics

**Core idea:** Incorporate document metadata (category, timestamp, author) into topic modeling.  
Topics can vary by covariates: "How does topic prevalence differ by category?"

We simulate STM behavior by training separate models per group and comparing.  
*(True STM is available in R's `stm` package or Python's `tomotopy`)*

In [ ]:
try:
    import tomotopy as tp
    print(f'✅ tomotopy {tp.__version__} — true STM available')
    TOMOTOPY_AVAILABLE = True
except ImportError:
    print('⚠️  tomotopy not installed — using manual covariate analysis')
    TOMOTOPY_AVAILABLE = False

In [ ]:
if TOMOTOPY_AVAILABLE:
    # ── True STM with tomotopy ─────────────────────────────────────────
    # Metadata: category labels as document-level covariate
    mdl = tp.PLDAModel(tw=tp.TermWeight.IDF, k=4, seed=42)

    for doc_tokens, label in zip(tokenized, labels):
        valid_tokens = [t for t in doc_tokens if t in dictionary.token2id]
        if valid_tokens:
            mdl.add_doc(words=valid_tokens)

    for i in range(0, 100, 10):
        mdl.train(10)
    print(f'STM (PLDA) training complete')

    # Show topics
    print('\nSTM Topics:')
    for k in range(mdl.k):
        words = ', '.join([w for w, _ in mdl.get_topic_words(k, top_n=8)])
        print(f'  Topic {k}: {words}')

else:
    # ── Manual covariate analysis ─────────────────────────────────────
    print('Simulating STM: comparing topic distributions by category\n')

    # Train one LDA per category group
    category_models = {}
    for cat_id, cat_name in enumerate(label_names):
        cat_idx    = [i for i, l in enumerate(labels) if l == cat_id]
        cat_corpus = [bow_corpus[i] for i in cat_idx]
        if len(cat_corpus) < 50:
            continue
        m = LdaModel(
            corpus=cat_corpus, id2word=dictionary,
            num_topics=4, random_state=42, passes=8
        )
        category_models[cat_name] = m
    print(f'Trained {len(category_models)} category-level models')

In [ ]:
# ── Compare topic distributions across categories ────────────────
# For each category, get average topic weights of its documents
print('Average topic proportion by category (using full corpus model):')
print()

results = {}
for cat_id, cat_name in enumerate(label_names):
    cat_idx = [i for i, l in enumerate(labels) if l == cat_id]
    cat_corpus = [bow_corpus[i] for i in cat_idx]

    topic_probs = []
    for doc in cat_corpus:
        dist = dict(base_model.get_document_topics(doc, minimum_probability=0))
        topic_probs.append([dist.get(t, 0.0) for t in range(4)])

    avg = np.mean(topic_probs, axis=0)
    results[cat_name.split('.')[-1]] = avg  # short name

df_stm = pd.DataFrame(results, index=[f'Topic {i}' for i in range(4)]).T
print(df_stm.round(3).to_string())

In [ ]:
# ── Heatmap: topic prevalence by category (STM-style) ─────────────
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    df_stm, annot=True, fmt='.2f',
    cmap='YlOrRd', linewidths=0.5,
    ax=ax, cbar_kws={'label': 'Avg Topic Proportion'}
)
ax.set_title('Topic Prevalence by Category\n(STM-style Covariate Analysis)', fontsize=13)
ax.set_xlabel('Topic', fontsize=11)
ax.set_ylabel('Category', fontsize=11)
plt.tight_layout()
plt.show()
print('This is what true STM produces — how topics differ across groups.')

---
## 7. Dynamic Topic Modeling (DTM) — Topics Over Time

**Core idea:** Topics evolve — the vocabulary of a topic can change over time.  
DTM models P(word | topic, time) instead of a static P(word | topic).

We simulate DTM by splitting the corpus into time slices and comparing topic word distributions.

In [ ]:
# ── Simulate time slices by document index ────────────────────────
# In practice, you'd sort by actual date. Here we use sequential splits as proxy.
n_docs = len(bow_corpus)
n_slices = 4
slice_size = n_docs // n_slices

slices = {
    f'Period {i+1}': {
        'corpus': bow_corpus[i*slice_size:(i+1)*slice_size],
        'texts':  tokenized[i*slice_size:(i+1)*slice_size]
    }
    for i in range(n_slices)
}

for name, s in slices.items():
    print(f'{name}: {len(s["corpus"])} docs')

In [ ]:
# ── Train one LDA per time slice and compare topic words ─────────
print('Training time-slice models...')

slice_models = {}
for period, data in slices.items():
    m = LdaModel(
        corpus=data['corpus'], id2word=dictionary,
        num_topics=4, random_state=42, passes=8
    )
    slice_models[period] = m
    print(f'  {period}: ✅')

print()
print('Topic word evolution across time (Topic 0 words):')
print()
for period, model in slice_models.items():
    words = [w for w, _ in model.show_topic(0, topn=8)]
    print(f'{period}: {", ".join(words)}')

In [ ]:
# ── Visualize how topic 0's top words change over time ─────────────
TOP_WORDS_TRACK = 15

# Build a matrix: for each word, what's its probability in each time slice?
# Focus on Topic 0
all_topic_words = set()
word_time_probs = defaultdict(dict)

for period, model in slice_models.items():
    topic_dist = dict(model.show_topic(0, topn=TOP_WORDS_TRACK))
    all_topic_words |= set(topic_dist.keys())
    for w, p in topic_dist.items():
        word_time_probs[w][period] = p

# Track the top 8 words by their max probability across all periods
top_tracked = sorted(all_topic_words,
    key=lambda w: max(word_time_probs[w].values()),
    reverse=True
)[:8]

periods = list(slices.keys())
fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.tab10.colors
for j, word in enumerate(top_tracked):
    probs = [word_time_probs[word].get(p, 0.0) for p in periods]
    ax.plot(periods, probs, 'o-', label=word,
            color=colors[j % 10], linewidth=2, markersize=8)

ax.set_xlabel('Time Period', fontsize=12)
ax.set_ylabel('Word Probability in Topic', fontsize=12)
ax.set_title('DTM-style: How Topic 0 Word Weights Evolve Over Time', fontsize=13)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. Author-Topic Model

**Core idea:** Assign topic distributions to **authors** (or sources), not just documents.  
Each author writes about some mix of topics — the model learns this from their combined documents.

In our dataset, we use the **newsgroup category as the "author"** — a proxy for information source.

In [ ]:
# ── Build author2doc mapping ──────────────────────────────────────
# In practice: {author_name: [doc_idx, doc_idx, ...]}
# Here: category name → list of document indices

author2doc = defaultdict(list)
for doc_idx, label in enumerate(labels):
    author_name = label_names[label].replace('.', '_')
    author2doc[author_name].append(doc_idx)

print('Author → document count:')
for author, doc_ids in sorted(author2doc.items()):
    print(f'  {author:35s}: {len(doc_ids)} documents')

In [ ]:
# ── Train Author-Topic Model ──────────────────────────────────────
at_model = AuthorTopicModel(
    corpus=bow_corpus,
    id2word=dictionary,
    num_topics=4,
    author2doc=dict(author2doc),
    random_state=42,
    passes=10
)

print('✅ Author-Topic Model trained')
print()

# Show topic distribution per author
print('Topic distribution by author (information source):')
print()
for author in sorted(author2doc.keys()):
    topic_dist = at_model[author]
    # Sort by probability
    topic_dist_sorted = sorted(topic_dist, key=lambda x: x[1], reverse=True)
    top2 = [f'T{t}({p:.2f})' for t, p in topic_dist_sorted[:2]]
    print(f'  {author:35s}: dominant topics → {", ".join(top2)}')

In [ ]:
# ── Visualize author-topic distributions ──────────────────────────
authors = sorted(author2doc.keys())
topic_matrix = np.zeros((len(authors), at_model.num_topics))

for i, author in enumerate(authors):
    dist = dict(at_model[author])
    for t in range(at_model.num_topics):
        topic_matrix[i, t] = dist.get(t, 0.0)

fig, ax = plt.subplots(figsize=(10, 5))
df_author = pd.DataFrame(
    topic_matrix,
    index=[a.split('_')[-1] for a in authors],  # short names
    columns=[f'Topic {t}' for t in range(at_model.num_topics)]
)

df_author.plot(kind='bar', ax=ax, stacked=True,
               color=['#028090', '#1A5276', '#D35400', '#6C3483'],
               edgecolor='white', linewidth=0.5)
ax.set_xlabel('Source (Author)', fontsize=12)
ax.set_ylabel('Topic Proportion', fontsize=12)
ax.set_title('Author-Topic Model: Topic Distribution per Information Source', fontsize=13)
ax.legend(loc='upper right')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Who writes about a topic most? ────────────────────────────────
print('Which author most discusses each topic?')
for t in range(at_model.num_topics):
    # Most probable word for this topic
    top_word = at_model.show_topic(t, topn=1)[0][0]
    best_author = authors[topic_matrix[:, t].argmax()]
    best_prop   = topic_matrix[:, t].max()
    print(f'  Topic {t} (key word: {top_word:15s}) → dominant author: {best_author} ({best_prop:.2f})')

---
## 9. Summary — When to Use Each Advanced Method

| Method | Use When |
|---|---|
| **Online LDA** | Corpus doesn't fit in memory, or data arrives as a stream |
| **Guided LDA** | You have domain knowledge; want predictable, stakeholder-friendly topics |
| **GSDMM** | Texts are very short (< 20 words): tweets, titles, headlines |
| **STM** | Document metadata exists and topics may vary by group, time, or demographics |
| **DTM** | You need to track how topic *vocabulary* changes over time (not just prevalence) |
| **Author-Topic** | You want to profile information sources, authors, or entities by their topic distribution |

**Key evaluation principles:**
- Coherence (C_v) > 0.5 — good topic quality
- Diversity > 0.7 — topics are distinct from each other  
- Stability > 0.5 — model is consistent across random seeds

**Next:** See Notebook 3 for BERTopic, Top2Vec, CTM, and modern embedding-based methods.